In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Premiers pas avec la découpe de circuits par coupures de fil


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
qiskit-addon-cutting~=0.10.0
```
</details>
Ce guide présente un exemple concret de coupures de fil avec le package `qiskit-addon-cutting`. Il explique comment reconstruire les valeurs d'espérance d'un circuit à sept qubits à l'aide de la technique de coupure de fil.

Une coupure de fil est représentée dans ce package par une instruction à deux qubits [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move), définie comme une réinitialisation du second qubit sur lequel l'instruction agit, suivie d'un échange des deux qubits. Cette opération équivaut à transférer l'état du premier qubit vers le second, tout en rejetant simultanément l'état entrant du second qubit.

Le package est conçu pour être cohérent avec la façon dont les coupures de fil doivent être traitées sur des qubits physiques. Par exemple, une coupure de fil peut transférer l'état du qubit physique $n$ et le continuer sur le qubit physique $m$ après la coupure. On peut voir la « découpe d'instructions » comme un cadre unifié permettant de considérer à la fois les coupures de fil et de portes dans le même formalisme (puisqu'une coupure de fil n'est qu'une instruction [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) coupée). Ce cadre permet également la réutilisation des qubits, ce qui est expliqué dans la section sur la [découpe manuelle des fils](#cut-wires-using-the-low-level-move-instruction).

L'instruction à un qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) offre une interface plus abstraite et simplifiée pour travailler avec les coupures de fil. Elle te permet d'indiquer à un niveau élevé où couper un fil dans le circuit, et l'addon de découpe de circuits insère automatiquement les instructions [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) appropriées.

L'exemple suivant illustre la reconstruction de la valeur d'espérance après une coupure de fil. Tu vas créer un circuit avec plusieurs portes non locales et définir des observables à estimer.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_ibm_runtime import SamplerV2, Batch
from qiskit_aer.primitives import EstimatorV2

from qiskit_addon_cutting.instructions import Move, CutWire
from qiskit_addon_cutting import (
    partition_problem,
    generate_cutting_experiments,
    cut_wires,
    expand_observables,
    reconstruct_expectation_values,
)


qc_0 = QuantumCircuit(7)
for i in range(7):
    qc_0.rx(np.pi / 4, i)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)
qc_0.cx(3, 4)
qc_0.cx(3, 5)
qc_0.cx(3, 6)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)

# Define observable
observable = SparsePauliOp(["ZIIIIII", "IIIZIII", "IIIIIIZ"])

# Draw circuit
qc_0.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg)

## Couper les fils avec l'instruction haut niveau `CutWire`
Ensuite, effectue des coupures de fil avec l'instruction à un qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) sur le qubit $q_3$. Une fois les sous-expériences prêtes à être exécutées, utilise la fonction [`cut_wires()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#cut_wires) pour transformer les instructions `CutWire` en instructions [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) sur des qubits nouvellement alloués.

In [2]:
qc_1 = QuantumCircuit(7)
for i in range(7):
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(CutWire(), [3])
qc_1.cx(3, 4)
qc_1.cx(3, 5)
qc_1.cx(3, 6)
qc_1.append(CutWire(), [3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg)

> **Info:** Lorsqu'un circuit est étendu par une ou plusieurs coupures de fil, l'observable doit être mis à jour pour tenir compte des qubits supplémentaires introduits. Le package `qiskit-addon-cutting` fournit une fonction utilitaire [`expand_observables()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#expand_observables), qui prend en argument des objets `PauliList` ainsi que les circuits original et étendu, et renvoie un nouveau `PauliList`.
> 
>     Ce `PauliList` renvoyé ne contiendra aucune information sur les coefficients de l'observable d'origine, mais ceux-ci peuvent être ignorés jusqu'à la reconstruction de la valeur d'espérance finale.

In [3]:
# Transform CutWire instructions to Move instructions
qc_2 = cut_wires(qc_1)

# Expand the observable to match the new circuit size
expanded_observable = expand_observables(observable.paulis, qc_0, qc_2)
print(f"Expanded Observable: {expanded_observable}")
qc_2.draw("mpl")

Expanded Observable: ['ZIIIIIIII', 'IIIZIIIII', 'IIIIIIIIZ']


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d398b397-0167-4db9-97ae-6ea502dc43e3-1.svg" alt="Output of the previous code cell" />

### Partition the circuit and observable

Now the problem can be separated into partitions. This is accomplished using the [`partition_problem()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#partition_problem) function with an optional set of partition labels to specify how to separate the circuit. Qubits sharing a common partition label are grouped together, and any non-local gates spanning more than one partition are cut.

If no partition labels are provided, then the partitioning will be automatically determined based on the connectivity of the circuit. Read the next section on [cutting wires manually](#cut-wires-using-the-low-level-move-instruction) for more information on including partition labels.

In [4]:
partitioned_problem = partition_problem(
    circuit=qc_2,
    observables=expanded_observable,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits[0].draw("mpl")

Subobservables to measure: 
{0: PauliList(['IIIII', 'ZIIII', 'IIIIZ']), 1: PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/5fb034f2-da8a-4f4d-ab9b-c990593e04fc-1.svg" alt="Output of the previous code cell" />

In [5]:
subcircuits[1].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg" alt="Output of the previous code cell" />

In this partitioning scheme, you have cut two wires, resulting in a sampling overhead of $4^4$.

### Generate subexperiments to execute and post-process results

To estimate the expectation value of the full-sized circuit, several subexperiments are generated from the decomposed gates' joint quasi-probability distribution and then executed on one (or more) QPUs. The [`generate_cutting_experiments`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) method does this by ingesting arguments for the `subcircuits` and `subobservables` dictionaries you created above, as well as for the number of samples to take from the distribution.

<Admonition type="note" title="Note about the number of samples">
    The `num_samples` argument specifies how many samples to draw from the quasi-probability distribution and determines the accuracy of the coefficients used for the reconstruction. Passing infinity (`np.inf`) ensures all coefficients are calculated exactly. Read the API docs on [generating weights](/docs/api/qiskit-addon-cutting/qpd#generate_qpd_weights) and [generating cutting experiments](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) for more information.
</Admonition>

In [6]:
# Generate subexperiments
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=np.inf
)

# Set a backend to use and transpile the subexperiments
backend = FakeManilaV2()
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Submit each partition's subexperiments to the Qiskit Runtime Sampler
# primitive, in a single batch so that the jobs will run back-to-back.
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
    # Retrieve results
    results = {label: job.result() for label, job in jobs.items()}

Lastly, the expectation value of the full circuit can be reconstructed using the [`reconstruct_expectation_values()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) method.


The code block below reconstructs the results and compares them with the exact expectation value.

In [7]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
# Apply the coefficients of the original observable
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


# Compute the exact expectation value using the `qiskit_aer` package.
estimator = EstimatorV2()
exact_expval = estimator.run([(qc_0, observable)]).result()[0].data.evs
print(
    f"Reconstructed expectation value: {np.real(np.round(reconstructed_expval, 8))}"
)
print(f"Exact expectation value: {np.round(exact_expval, 8)}")
print(
    f"Error in estimation: {np.real(np.round(reconstructed_expval-exact_expval, 8))}"
)
print(
    f"Relative error in estimation: {np.real(np.round((reconstructed_expval-exact_expval) / exact_expval, 8))}"
)

Reconstructed expectation value: 1.45965266
Exact expectation value: 1.59099026
Error in estimation: -0.1313376
Relative error in estimation: -0.08255085


![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg)

Dans ce schéma de partitionnement, tu as coupé deux fils, ce qui entraîne un surcoût d'échantillonnage de $4^4$.

### Générer les sous-expériences à exécuter et post-traiter les résultats

Pour estimer la valeur d'espérance du circuit complet, plusieurs sous-expériences sont générées à partir de la distribution quasi-probabiliste conjointe des portes décomposées, puis exécutées sur un ou plusieurs QPU. La méthode [`generate_cutting_experiments`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) réalise cela en prenant en argument les dictionnaires `subcircuits` et `subobservables` créés précédemment, ainsi que le nombre d'échantillons à tirer de la distribution.

> **Note:** L'argument `num_samples` spécifie le nombre d'échantillons à tirer de la distribution quasi-probabiliste et détermine la précision des coefficients utilisés pour la reconstruction. Passer l'infini (`np.inf`) garantit que tous les coefficients sont calculés exactement. Consulte la documentation de l'API sur la [génération des poids](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qpd#generate_qpd_weights) et la [génération des expériences de découpe](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) pour plus d'informations.

In [8]:
qc_1 = QuantumCircuit(8)
for i in [*range(4), *range(5, 8)]:
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(Move(), [3, 4])
qc_1.cx(4, 5)
qc_1.cx(4, 6)
qc_1.cx(4, 7)
qc_1.append(Move(), [4, 3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

# Expand observable
observable_expanded = SparsePauliOp(["ZIIIIIII", "IIIIZIII", "IIIIIIIZ"])
qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/15461a2c-85a9-4cb2-a632-b9597ccbc4bd-0.svg" alt="Output of the previous code cell" />

Pour terminer, la valeur d'espérance du circuit complet peut être reconstruite à l'aide de la méthode [`reconstruct_expectation_values()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values).

Le bloc de code ci-dessous reconstruit les résultats et les compare à la valeur d'espérance exacte.

In [9]:
partitioned_problem = partition_problem(
    circuit=qc_1,
    partition_labels="AAAABBBB",
    observables=observable_expanded.paulis,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits["A"].draw("mpl")

Subobservables to measure: 
{'A': PauliList(['IIII', 'ZIII', 'IIIZ']), 'B': PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/2139745a-bdc3-40bd-bd6f-d26d2a5b5b14-1.svg" alt="Output of the previous code cell" />

In [10]:
subcircuits["B"].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/4aeb3f1f-a55e-49c4-a7bd-837132429ee1-0.svg" alt="Output of the previous code cell" />

> **Caution:** Pour reconstruire avec précision la valeur d'espérance, les coefficients de l'observable d'origine (qui diffèrent de la sortie de `generate_cutting_experiments()`) doivent être appliqués au résultat de la reconstruction, car cette information a été perdue lors de la génération des expériences de découpe ou lors de l'expansion de l'observable.
> 
>     Ces coefficients peuvent généralement être appliqués via `numpy.dot()` comme illustré précédemment.
## Couper les fils avec l'instruction bas niveau `Move`
L'une des limitations de l'instruction haut niveau `CutWire` est qu'elle ne permet pas la réutilisation des qubits. Si cela est souhaité pour une expérience de découpe, tu peux à la place placer manuellement des instructions [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move). Cependant, comme l'instruction `Move` rejette l'état du qubit de destination, il est important que ce qubit ne partage aucun enchevêtrement avec le reste du système. Sinon, l'opération de réinitialisation provoquera un effondrement partiel de l'état du circuit après la coupure de fil.

Le bloc de code ci-dessous effectue une coupure de fil sur le qubit $q_3$ pour le même exemple de circuit que précédemment. La différence ici est que tu peux réutiliser un qubit en inversant l'opération `Move` là où la seconde coupure de fil a été effectuée (cela n'est cependant pas toujours possible et dépend du circuit à couper).